# ECCT on LDPC(49,24)

This notebook clones the repository, checks the Kaggle GPU, trains ECCT on the same LDPC code and model dimensions as AECCT, and prints the resulting BER/FER log.

Enable **Internet** and a **GPU accelerator** in Kaggle before running. This faster comparison uses 500 ECCT epochs and 500 epochs per AECCT phase. Increase ECCT to 1,000 epochs per phase-level comparison or 2,000 epochs for the full original AECCT training budget.

In [1]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/gouravanirudh05/SRIP_LDPC_Decoding_using_Machine_Learning.git'
REPO_DIR = Path('/kaggle/working/ldpc_repo')

if not (REPO_DIR / '.git').exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

ECCT_DIR = REPO_DIR / 'ECCT'
if not (ECCT_DIR / 'Main.py').exists():
    raise FileNotFoundError('ECCT/Main.py is missing from the cloned repository.')

os.chdir(ECCT_DIR)
print('Working directory:', Path.cwd())

Cloning into '/kaggle/working/ldpc_repo'...


Working directory: /kaggle/working/ldpc_repo/ECCT


In [2]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'einops', 'tqdm'], check=True)

import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU is not available. In Kaggle, select GPU under Notebook options.')
print('GPU:', torch.cuda.get_device_name(0))

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


## Training configuration

The code, rate, optimizer, batch size, seed, number of blocks, and embedding dimension match the AECCT run. ECCT is run for 500 epochs for this faster preliminary comparison.

In [3]:
import time

ECCT_EPOCHS = 500
command = [
    sys.executable, 'Main.py',
    '--gpus=0',
    f'--epochs={ECCT_EPOCHS}',
    '--workers=4',
    '--lr=1e-4',
    '--batch_size=128',
    '--test_batch_size=2048',
    '--seed=42',
    '--code_type=LDPC',
    '--code_n=49',
    '--code_k=24',
    '--N_dec=6',
    '--d_model=128',
    '--h=8',
]
print('Running:', ' '.join(command))
start = time.perf_counter()
subprocess.run(command, cwd=str(ECCT_DIR), check=True)
print(f'Total wall time: {(time.perf_counter() - start) / 3600:.2f} hours')

Running: /usr/bin/python3 Main.py --gpus=0 --epochs=500 --workers=4 --lr=1e-4 --batch_size=128 --test_batch_size=2048 --seed=42 --code_type=LDPC --code_n=49 --code_k=24 --N_dec=6 --d_model=128 --h=8


Path to model/logs: Results_ECCT/LDPC__Code_n_49_k_24__27_07_2026_04_24_37
Namespace(epochs=500, workers=4, lr=0.0001, gpus='0', batch_size=128, test_batch_size=2048, seed=42, code_type='LDPC', code_k=24, code_n=49, standardize=False, N_dec=6, d_model=128, h=8, code=<__main__.Code object at 0x7e867105c230>, path='Results_ECCT/LDPC__Code_n_49_k_24__27_07_2026_04_24_37')
Self-Attention Sparsity Ratio=72.26%, Self-Attention Complexity Ratio=13.87%
Mask:
 tensor([[[[False,  True,  True,  ...,  True,  True,  True],
          [ True, False,  True,  ...,  True,  True,  True],
          [ True,  True, False,  ...,  True,  True,  True],
          ...,
          [ True,  True,  True,  ..., False,  True,  True],
          [ True,  True,  True,  ...,  True, False,  True],
          [ True,  True,  True,  ...,  True,  True, False]]]])
ECC_Transformer(
  (decoder): Encoder(
    (layers): ModuleList(
      (0-5): 6 x EncoderLayer(
        (self_attn): MultiHeadedAttention(
          (linears): Module

FER count threshold reached for EbN0:4
Test EbN0=4, BER=2.61e-03
FER count threshold reached for EbN0:5
Test EbN0=5, BER=2.07e-04



Test Loss 4: 6.93e-03 5: 5.76e-04 6: 2.59e-05
Test FER 4: 3.00e-02 5: 3.07e-03 6: 1.46e-04
Test BER 4: 2.61e-03 5: 2.07e-04 6: 8.01e-06
Test -ln(BER) 4: 5.95e+00 5: 8.48e+00 6: 1.17e+01
# of testing samples: [100352.0, 100352.0, 698368.0]
 Test Time 373.0766406059265 s



FER count threshold reached for EbN0:6
Test EbN0=6, BER=8.01e-06
Total wall time: 9.88 hours


In [4]:
# Print the latest ECCT result directory and its final log lines.
result_dirs = sorted((ECCT_DIR / 'Results_ECCT').glob('*'), key=lambda p: p.stat().st_mtime)
if not result_dirs:
    raise FileNotFoundError('No ECCT result directory was produced.')
latest = result_dirs[-1]
log_file = latest / 'logging.txt'
print('Result directory:', latest)
print('Checkpoint:', latest / 'best_model')
print('\n'.join(log_file.read_text(errors='replace').splitlines()[-40:]))

Result directory: /kaggle/working/ldpc_repo/ECCT/Results_ECCT/LDPC__Code_n_49_k_24__27_07_2026_04_24_37
Checkpoint: /kaggle/working/ldpc_repo/ECCT/Results_ECCT/LDPC__Code_n_49_k_24__27_07_2026_04_24_37/best_model
Training epoch 493, Batch 500/1000: LR=1.06e-06, Loss=2.58e-02 BER=1.05e-02 FER=9.48e-02
Training epoch 493, Batch 1000/1000: LR=1.06e-06, Loss=2.60e-02 BER=1.06e-02 FER=9.58e-02
Epoch 493 Train Time 70.01990723609924s

Training epoch 494, Batch 500/1000: LR=1.05e-06, Loss=2.66e-02 BER=1.09e-02 FER=9.73e-02
Training epoch 494, Batch 1000/1000: LR=1.05e-06, Loss=2.66e-02 BER=1.09e-02 FER=9.71e-02
Epoch 494 Train Time 70.267897605896s

Training epoch 495, Batch 500/1000: LR=1.04e-06, Loss=2.64e-02 BER=1.08e-02 FER=9.77e-02
Training epoch 495, Batch 1000/1000: LR=1.04e-06, Loss=2.64e-02 BER=1.08e-02 FER=9.70e-02
Epoch 495 Train Time 70.69523119926453s

Training epoch 496, Batch 500/1000: LR=1.02e-06, Loss=2.67e-02 BER=1.09e-02 FER=9.73e-02
Training epoch 496, Batch 1000/1000: LR=

In [5]:
from pathlib import Path
import tarfile

# For ECCT:
result_root = Path("/kaggle/working/ldpc_repo/ECCT/Results_ECCT")

# For AECCT, use instead:
# result_root = Path("/kaggle/working/ldpc_repo/AECCT-main/logs/Results_AECCT")

result_dirs = sorted(result_root.glob("*"), key=lambda p: p.stat().st_mtime)
latest = result_dirs[-1]

archive = Path("/kaggle/working/LDPC49_results.tar.gz")

with tarfile.open(archive, "w:gz") as tar:
    tar.add(latest, arcname=latest.name)

print("Saved:", archive)
print("Log:", latest / "logging.txt")
print("Checkpoint:", latest / "best_model")

Saved: /kaggle/working/LDPC49_results.tar.gz
Log: /kaggle/working/ldpc_repo/ECCT/Results_ECCT/LDPC__Code_n_49_k_24__27_07_2026_04_24_37/logging.txt
Checkpoint: /kaggle/working/ldpc_repo/ECCT/Results_ECCT/LDPC__Code_n_49_k_24__27_07_2026_04_24_37/best_model
